# Metric-Guided Content Anonymization — Baseline Model

## 1. Objective

The objective is to improve the content preservation of the LLM-based speech anonymization pipeline proposed in *Content Anonymization for Privacy in Long-form Audio* (Aggazzotti et al.), while maintaining its privacy benefits.

The central idea is to use three complementary text-based metrics to guide the paraphrasing process:

- **Semantic Similarity** measures global semantic proximity between the original and paraphrased texts.
- **Term Preservation Index (TPI)** measures the preservation of relevant terms.
- **Natural Language Inference (NLI)** estimates whether the paraphrased text preserves the information conveyed by the original text.

Rather than relying exclusively on the initial LLM prompt, the proposed method evaluates generated paraphrases and selectively requests revisions when content preservation appears insufficient.

The first implementation will use an existing LLM without fine-tuning. Fine-tuning will be considered in a subsequent stage, once the baseline has been implemented and evaluated.

## 2. Proposed Pipeline

### Step 1 — Contextual paraphrasing

The original transcript is passed to an LLM using the contextual paraphrasing strategy of the reference paper. The model is instructed to modify linguistic style while preserving the original meaning, facts, and relevant information.

The contextual nature of the paraphrasing process must be maintained because stylistic characteristics can extend across multiple utterances.

### Step 2 — Content evaluation

The generated transcript is compared with the original transcript using three metrics:

| Metric | Purpose |
|---|---|
| Semantic Similarity | Assess global semantic preservation |
| TPI | Detect the loss of relevant terms |
| NLI | Detect potential changes in entailment or meaning |

These metrics provide complementary signals. No individual metric is assumed to be a perfect measure of semantic preservation.

### Step 3 — Candidate acceptance or revision

The system uses the metric scores to determine whether a generated paraphrase requires further attention.

- **Accept:** If all calibrated acceptance criteria are satisfied, retain the generated transcript.
- **Revise:** If one or more criteria are not satisfied, ask the LLM to revise the candidate, targeting the detected content-preservation issues.
- **Re-evaluate:** Recalculate all three metrics after revision.
- **Stop:** Accept the revised candidate if the criteria are satisfied, or stop after a predefined maximum number of attempts.

If the source transcript is unreliable or heavily corrupted by ASR errors, the system should flag the example rather than automatically reconstructing potentially unsupported content.

The acceptance thresholds and stopping criteria will be calibrated empirically rather than selected arbitrarily.

## 3. Metric-Guided Revision

The revision prompt will receive:

1. The original transcript.
2. The current paraphrased transcript.
3. The evaluation scores.
4. A description of the detected content-preservation issues.

The LLM will be instructed to make the minimum necessary changes to restore the original meaning and relevant information, while retaining the stylistic modifications required for anonymization.

The objective is **not to maximize the three metrics independently**. Excessive similarity to the original may preserve identifying linguistic habits and undermine the privacy objective.

The baseline will use the metrics as evaluation and decision signals, not as a differentiable loss function. Direct optimization of the LLM using a reward or loss will be explored in a later fine-tuning stage.

## 4. Evaluation Protocol

The proposed system will be compared with the original paraphrasing baseline under the same experimental conditions.

The following variants will be evaluated:

1. **Original baseline:** Contextual LLM paraphrasing without metric-guided revision.
2. **Metric evaluation only:** Apply the three metrics without modifying the generated text.
3. **Metric-guided revision:** Trigger targeted LLM revisions when the acceptance criteria are not satisfied.

The main evaluation metrics will be:

- **Greedy Alignment Score (GAS):** Higher is better.
- **DTW Similarity:** Higher is better.
- **Semantic Similarity:** Higher is generally better, subject to the limitations of embedding-based metrics.
- **TPI:** Higher indicates greater preservation of the relevant terms.
- **NLI:** Examine entailment, neutral, and contradiction probabilities rather than relying only on the predicted label.

Privacy will be evaluated independently using the content-based speaker re-identification attack and Equal Error Rate (EER), following the reference paper's protocol where possible. The goal is to improve content preservation without sacrificing the privacy benefits of paraphrasing.

The evaluation will also track the number of revision attempts, LLM calls, computational cost, and the frequency of cases requiring intervention.

A manually annotated subset will be used to verify whether the metrics correctly identify meaningful content losses and whether revisions genuinely improve the text.

## 5. Experimental Hypotheses

The baseline will test the following hypotheses:

- **H1:** Metric-guided revision improves GAS and DTW Similarity compared with the original paraphrasing baseline.
- **H2:** Combining Semantic Similarity, TPI, and NLI detects content-preservation issues more effectively than relying on a single metric.
- **H3:** Selective revision achieves comparable or better content preservation than unconditional additional paraphrasing, with fewer LLM calls.
- **H4:** The improvements in content preservation do not materially degrade the privacy performance of the anonymization pipeline.

These hypotheses will be assessed on held-out data, with consistent generation settings and explicit comparisons against the original baseline.

## 6. Implementation Strategy

The implementation will proceed in the following order:

1. Reproduce and document the original paraphrasing baseline.
2. Integrate Semantic Similarity, TPI, and NLI into the evaluation pipeline.
3. Analyze metric disagreements and establish a manually validated evaluation subset.
4. Calibrate acceptance thresholds and implement the decision logic.
5. Implement targeted LLM revision with a maximum attempt budget.
6. Compare the original and revised pipelines using GAS, DTW Similarity, privacy EER, and computational cost.
7. Document failure cases, limitations, and reproducibility details.

The implementation should remain modular so that the generation strategy, metrics, decision rules, and revision mechanism can be tested independently.

## 7. Future Work — Fine-Tuning

Once the baseline has been evaluated, the next stage will investigate whether the same content-preservation objectives can be integrated more directly into model optimization.

Potential directions include:

- Training or fine-tuning an open-weight LLM using examples of successful paraphrases.
- Defining a reward function combining Semantic Similarity, TPI, NLI, and a privacy-related objective.
- Exploring preference optimization or reinforcement-learning-based methods.
- Evaluating whether a learned model can achieve the same or better preservation with fewer revision calls.

The three metrics will not automatically be treated as a reliable reward function. Their correlation with human judgments must first be established, and safeguards will be needed to prevent the model from exploiting weaknesses in the metrics.

## 8. Expected Contribution

The proposed baseline introduces a metric-guided revision mechanism into an existing LLM-based content anonymization pipeline.

Its intended contribution is to determine whether selective, targeted revisions can improve semantic preservation while retaining the privacy benefits of contextual stylistic rewriting.

The baseline will provide the experimental foundation for a subsequent fine-tuning approach, in which content preservation and privacy objectives may be incorporated more directly into model optimization.
